# Falcon 9 First-Stage Landing: Exploratory Data Analysis

This notebook explores the project's **cleaned, labeled historical dataset**. It compares labeled landing outcomes, looks at success rates across launch years, launch sites, and booster families, and saves charts to `reports/figures/`.

> **Data scope:** This is a community-compiled historical dataset with records through December 18, 2021. It is not a current SpaceX dataset. Only records with explicit binary landing outcomes are included here; records with ambiguous or missing outcomes were excluded during the previous cleaning stage. These descriptive comparisons do not show that a year, site, or booster version caused a landing outcome.
>
> **Plotting note:** During the initial EDA run, Windows Application Control blocked Pandas' native DLL while Seaborn was importing it. To keep the notebook runnable without bypassing that security policy, the charts use Matplotlib directly. A later check with system Python 3.15.1 found that Pandas imports successfully; Seaborn was not rechecked.

## 1. Load the data

The helper module reads the cleaned CSV, checks required columns and binary labels, parses the launch date, and groups booster versions into broad families. The observations and charts below come from the CSV on disk, not sample or generated data.

In [ ]:
from pathlib import Path
import csv
import sys
from datetime import datetime

import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd()
while not (project_root / "data" / "processed" / "falcon9_launches_labeled.csv").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("Open this notebook from the project workspace.")
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))
from eda import build_summaries, create_visualizations, load_data

data_path = project_root / "data" / "processed" / "falcon9_launches_labeled.csv"
figures_path = project_root / "reports" / "figures"
with data_path.open(encoding="utf-8-sig", newline="") as source_file:
    source_columns = csv.DictReader(source_file).fieldnames
data = load_data(data_path)
summaries = build_summaries(data)

print(f"Loaded {len(data)} labeled Falcon 9 records.")
print(f"Source columns ({len(source_columns)}): {', '.join(source_columns)}")

## 2. Inspect columns and missing values

The source includes descriptive fields such as date, launch site, booster version, landing outcome, and the prepared `landing_success` label. Blank values in other fields are counted as missing; this notebook does not fill them with guesses.

In [ ]:
data[0]

In [ ]:
missing_counts = {
    column: sum(not (row[column] or "").strip() for row in data)
    for column in source_columns
}
print("Missing values by column:", missing_counts)
row_keys = {tuple(sorted(row.items())) for row in data}
print(f"Exact duplicate rows in the labeled data: {len(data) - len(row_keys)}")
dates = []
for row in data:
    try:
        dates.append(datetime.strptime(row["Date"], "%d-%m-%Y").date())
    except ValueError:
        pass
print(f"Valid date range: {min(dates)} to {max(dates)}")
print(f"Rows with missing or malformed dates (excluded from year trend): {len(data) - len(dates)}")

## 3. Successful versus unsuccessful landings

`landing_success` is 1 for an explicitly successful landing and 0 for an explicitly unsuccessful landing, following the cleaning rules. Class counts are shown alongside the overall rate; the data is imbalanced, so the percentage should not be read without its counts.

In [ ]:
print("Landing outcome counts:", summaries["outcomes"])
success_count = sum(int(row["landing_success"]) for row in data)
unsuccessful_count = len(data) - success_count
print(f"Successful: {success_count} ({success_count / len(data):.1%})")
print(f"Unsuccessful: {unsuccessful_count} ({unsuccessful_count / len(data):.1%})")

## 4. Landing success over time

The line shows the share of labeled launches with successful landings in each year. Each point is annotated with that year's launch count (`n`), which helps show when a rate is based on only a few records.

In [ ]:
print(f"Records excluded from the year trend because the date is invalid: {summaries['invalid_date_count']}")
for year, summary in summaries["by_year"].items():
    print(f"{year}: {summary['successful_landings']}/{summary['launches']} successful "
          f"({summary['success_rate']:.1%})")
print("Yearly landing rates are descriptive; small yearly counts make them uncertain.")

## 5. Compare launch sites and booster families

A booster family is grouped from the version prefix (for example, `F9 B5`); individual serial numbers are not treated as separate versions. Site and booster rates include the number of records in each group. Small groups can have unstable rates.

In [ ]:
for title, key in (("Launch site", "by_site"), ("Booster family", "by_booster_family")):
    print(title)
    for category, summary in summaries[key].items():
        print(f"{category}: {summary['successful_landings']}/{summary['launches']} successful "
              f"({summary['success_rate']:.1%})")

## 6. Visualizations

The following figures are generated directly from the loaded CSV. They are also saved as PNG files in `reports/figures/`.

In [ ]:
figures = create_visualizations(data, figures_path)
for filename, figure in figures:
    print(filename)
    display(figure)
    plt.close(figure)

### Landing outcome counts

![Landing outcome counts](../reports/figures/landing_outcomes.png)

### Success rates over time

![Success rate by year](../reports/figures/landing_success_by_year.png)

### Success rates by launch site

![Success rate by launch site](../reports/figures/landing_success_by_site.png)

### Success rates by booster family

![Success rate by booster family](../reports/figures/landing_success_by_booster_family.png)

## 7. Findings and limitations

Use the tables and charts above to compare the observed rates and counts. Keep interpretations descriptive: the dataset is small, historical, and has fewer unsuccessful examples than successful ones. In addition, the source is a community compilation, and ambiguous landing outcomes were excluded during labeling. These results are not model performance and should not be used to claim that a particular site, year, or booster family caused success.